# Vector Database Setup
This notebook guides you through the three key steps for setting up your vector database. By the end of this setup, you’ll have both an index and an indexer ready to use for loading and searching embeddings.


**Assumptions**

You have already uploaded your documents to the embeddings folder under a subfolder named after the index you want to create.
Example: `embeddings/bioformer/` should contain all of the documents for the `bioformer` index.

Each document file in the subfolder:

- Is 16MB or smaller (approximately 1,000 documents per file).

- Is in .jsonl format (JSON Lines).

- Files that exceed 16MB or are not in .jsonl format will not be loaded into the vector database or included in the index.

In [ ]:
pip install --upgrade azure-keyvault-secrets azure-identity azure-ai-ml azure-search-documents

## Set up

In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import AzureAISearchConnection
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes import SearchIndexerClient

# Authenticate to Key Vault
credential = DefaultAzureCredential()
key_vault = "dibbsttc6059789213"
secret_client = SecretClient(
    vault_url=f"https://{key_vault}.vault.azure.net/", credential=credential
)

# Get secrets
SUBSCRIPTION = secret_client.get_secret("subscription").value
RESOURCE_GROUP = secret_client.get_secret("resource-group").value
WS_NAME = secret_client.get_secret("workspace-name").value
DATASTORE_NAME = "workspaceblobstore"

AI_SEARCH_API_KEY = secret_client.get_secret("ai-search-api-key").value
AI_SEARCH_SERVICE_NAME = "dibbs-ttc-ai-search"
AI_SEARCH_SERVICE_ENDPOINT = f"https://{AI_SEARCH_SERVICE_NAME}.search.windows.net"

# Authenticate and connect to workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION,
    RESOURCE_GROUP,
    WS_NAME,
)

# Register the AI Search connection (metadata)
ai_search_connection = AzureAISearchConnection(
    name=AI_SEARCH_SERVICE_NAME,
    endpoint=AI_SEARCH_SERVICE_ENDPOINT,
    api_key=AI_SEARCH_API_KEY,
    description="Connection to Azure AI Search for vector indexing and queries",
)

ml_client.connections.create_or_update(ai_search_connection)

# Configure the Index Client, which is used for creating, updating, deleting, or listing indexes in AI Search
index_client = SearchIndexClient(
    endpoint=AI_SEARCH_SERVICE_ENDPOINT, credential=AzureKeyCredential(AI_SEARCH_API_KEY)
)

# Configure the Indexer Client, which is used for adding documents to the index in AI Search
indexer_client = SearchIndexerClient(
    endpoint=AI_SEARCH_SERVICE_ENDPOINT, credential=AzureKeyCredential(AI_SEARCH_API_KEY)
)

# EMBEDDING VARIABLES
EMBEDDINGS_FOLDER = "embeddings"

# Get Datastore information
datastore = ml_client.datastores.get(DATASTORE_NAME)
CONTAINER_NAME = datastore.account_name

# INDEX and INDEXER VARIABLES
INDEX_NAME = "e5basev2"
BLOB_CONN_STRING = secret_client.get_secret("blob-storage-connection-string").value
CONTAINER_NAME = secret_client.get_secret("blob-storage-container").value
CONTAINER_QUERY = f"embeddings/{INDEX_NAME}/"
RUN_INTERVAL = True
INTERVAL_MINUTES = 5

# HNSW INDEX VARIABLES
EF_CONSTRUCTION = 200
M_VALUE = 64
EF_SEARCH = 100
METRIC = "cosine"

## Step 1: Get the EMBEDDING_SIZE
This step retrieves the `EMBEDDING_SIZE` (aka the `vector_search_dimensions`) that is required for configuring the search index.


In [7]:
import json

from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate a file system over the workspace so we can interact with data
# assets directly--we get all the goodies like open, ls, etc.
DATASTORE_PATH = f"azureml://subscriptions/{SUBSCRIPTION}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WS_NAME}/datastores/{DATASTORE_NAME}"
fs = AzureMachineLearningFileSystem(DATASTORE_PATH)


def get_embedding_size(embedding_file: str) -> int:
    """
    Gets the embedding size from a specified file in the datastore.

    :param embedding_file: The path to the embedding file in the datastore
    :return: A int that describes the embedding size (vector dimensionality)
    """
    with fs.open(embedding_file, "r") as fp:
        line = fp.readline()
        d = json.loads(line)
    return len(d["descriptionVector"])

## Step 2: Configure HNSW Index
This step sets the schema for the HNSW index, constructing how the index should be built and how the index will be used during search.

In [5]:
from typing import Literal

from azure.search.documents.indexes.models import HnswAlgorithmConfiguration
from azure.search.documents.indexes.models import SearchableField
from azure.search.documents.indexes.models import SearchField
from azure.search.documents.indexes.models import SearchFieldDataType
from azure.search.documents.indexes.models import SearchIndex
from azure.search.documents.indexes.models import SimpleField
from azure.search.documents.indexes.models import VectorSearch
from azure.search.documents.indexes.models import VectorSearchProfile


def _set_up_hnsw_config(
    m_value: int = 64,
    ef_construction: int = 200,
    ef_search: int = 100,
    metric: Literal["cosine", "euclidean", "dotProduct", "hamming"] = "cosine",
) -> HnswAlgorithmConfiguration:
    """
    Sets up HNSW configuration for vector search.

    :param m_value: The number of bi-directional links created for every new element during construction.
    :param ef_construction: The size of the dynamic list containing the nearest neighbors, which is used during index time
    :param ef_search: The size of the dynamic list containing the nearest neighbors, which is used during search time.
    :param metric: The similarity metric to use for vector comparisons.
    :return: A configured HnswAlgorithmConfiguration object.
    """
    return HnswAlgorithmConfiguration(
        name="hnsw-vector-config",
        kind="hnsw",
        parameters={
            "m": m_value,
            "efConstruction": ef_construction,
            "efSearch": ef_search,
            "metric": metric,
        },
    )


def configure_search_index(index_name: str, vector_search_dimensions: int) -> None:
    """
    Configures and creates an Azure AI Search index with HNSW vector search.

    :index_name: The name of the search index.
    :vector_search_dimensions: The dimensionality of the embedding vectors.
    """

    # Define index fields
    fields = [
        SimpleField(
            name="id",
            type=SearchFieldDataType.String,
            key=True,
            filterable=True,
        ),
        SearchableField(
            name="description",
            type=SearchFieldDataType.String,
        ),
        SearchField(
            name="descriptionVector",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=vector_search_dimensions,
            vector_search_profile_name="vector-profile",
        ),
    ]

    # Set up HNSW configuration
    hnsw_config = _set_up_hnsw_config()

    # Define vector search settings
    vector_search = VectorSearch(
        algorithms=[hnsw_config],
        profiles=[
            VectorSearchProfile(
                name="vector-profile",
                algorithm_configuration_name="hnsw-vector-config",
            )
        ],
    )

    # Create and return the index
    index = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search,
    )

    # Add the configured index to the vector database via the index_client

    result = index_client.create_or_update_index(index)
    print(f" {result.name} created")

## Step 3: Configure Indexer
This step sets indexer for the index that was created in Step 3. The indexer points to the blob storage where the documents for the index are stored, inserting the documents into the vector database. 

Assumptions:
 - This step assumes you have already added the documents to the `embeddings` folder under a sub-folder that is the name of the index you'd like to use, e.g., `embeddings/bioformer` should contain all of the documents for the `bioformer` index. 
-  This step assumes that all of the documents in the subfolder are 16MB or smaller (approximately 1000 documents per file) and that the documents are `.jsonl` files. If the files are too large or not `.jsonl`, they will not be loaded into the vector database nor added to the index.

In [21]:
import datetime

from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import IndexingParameters
from azure.search.documents.indexes.models import IndexingParametersConfiguration
from azure.search.documents.indexes.models import SearchIndexer
from azure.search.documents.indexes.models import SearchIndexerDataSourceConnection


def create_or_update_blob_indexer(
    index_name: str,
    search_service_endpoint: str,
    search_service_api_key: str,
    blob_connection_string: str,
    container_name: str,
    container_query: str,
    run_interval: bool = False,
    interval_minutes: int = 5,
):
    """
    Creates or updates an Azure AI Search indexer that indexes data from an Azure Blob
    Storage container.

    :param index_name: The name of the search index to index data into
    :param search_service_endpoint: The endpoint URL of the Azure AI Search service
    :param search_service_api_key: The API key for the Azure AI Search service
    :param blob_connection_string: The connection string for the Azure Blob Storage account
    :param container_name: The name of the Blob Storage container
    :param container_query: The path within the container to index
    :param run_interval: Whether to run the indexer at regular intervals
    :param interval_minutes: The interval in minutes for the indexer to run, default is 5 minutes
    :return: None
    """
    # Initialize the Search Indexer Client
    indexer_client = SearchIndexerClient(
        endpoint=search_service_endpoint, credential=AzureKeyCredential(search_service_api_key)
    )

    # Create or update data source
    data_source = SearchIndexerDataSourceConnection(
        name=f"{index_name}-data-source",
        type="azureblob",
        connection_string=blob_connection_string,
        container={"name": container_name, "query": container_query},
    )
    indexer_client.create_or_update_data_source_connection(data_source)

    # Define indexing parameters
    indexing_parameters = IndexingParameters(
        configuration=IndexingParametersConfiguration(
            parsing_mode="jsonLines",
            query_timeout=None,
        )
    )

    # Create or update indexer
    if run_interval:
        schedule = {
            "interval": f"PT{interval_minutes}M",
            "startTime": f"{datetime.datetime.now(datetime.timezone.utc).isoformat()}",
        }
    indexer = SearchIndexer(
        name=f"{index_name}-indexer",
        data_source_name=f"{index_name}-data-source",
        target_index_name=index_name,
        parameters=indexing_parameters,
        schedule=schedule if run_interval else None,
    )

    indexer_client.create_or_update_indexer(indexer)

## Putting it together: Run steps 1-3 on index of your choice

In [ ]:
# Step 1
document = fs.ls(CONTAINER_QUERY)[0]
EMBEDDING_SIZE = get_embedding_size(embedding_file=document)

# Step 2
print(f"Configuring and creating index for {INDEX_NAME}...")
configure_search_index(index_name=INDEX_NAME, vector_search_dimensions=EMBEDDING_SIZE)

# Step 3
create_or_update_blob_indexer(
    index_name=INDEX_NAME,
    search_service_endpoint=AI_SEARCH_SERVICE_ENDPOINT,
    search_service_api_key=AI_SEARCH_API_KEY,
    blob_connection_string=BLOB_CONN_STRING,
    container_name=CONTAINER_NAME,
    container_query=CONTAINER_QUERY,
    run_interval=RUN_INTERVAL,
    interval_minutes=INTERVAL_MINUTES,
)

## Check the status of the indexer
status = indexer_client.get_indexer_status(f"{INDEX_NAME}-indexer")
print(f"Indexer Name: {status.name}")
print(f"Indexer Status: {status.status}")
last_run = status.last_result
print(f"Last Run Status: {last_run.status}")
print(f"Start Time: {last_run.start_time}")
print(f"End Time: {last_run.end_time}")
print(f"Error Count: {last_run.errors.count if last_run.errors else 0}")
print(f"Items Succeeded: {last_run.item_count}")
print(f"Items Failed: {last_run.failed_item_count}")